# Data Cleaning — Goods & Services Acquisitions (Q1 2026)

**Source:** `ADQUISICION_DE_BIENES_Y_SERVICIOS_ENERO_A_MARZO_2026.csv`
**Period covered:** January – March 2026
**Author:** _add your name_
**Last run:** _add date_

## Objective

Clean the raw quarterly procurement report and validate three data
quality issues identified during exploratory analysis:

1. Trailing/leading whitespace in provider names (`nombre_prov`)
2. Inconsistent null representations in the selection-process number
   field (`nro_proc_sel`) — mixed use of `NaN`, `"."`, and `"0"`
3. Negative amounts in `total_fact_moneda` — confirmed to correspond
   to legitimate business events (rebates/cancellations), not data
   entry errors

## Table of contents

1. [Setup](#1.-Setup)
2. [Load & clean the data](#2.-Load-%26-clean-the-data)
3. [Data quality validation](#3.-Data-quality-validation)
4. [Exploratory visualizations](#4.-Exploratory-visualizations)
5. [Export cleaned dataset](#5.-Export-cleaned-dataset)


## 1. Setup

In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# This notebook lives in notebooks/, the cleaning module is one level up in src/
sys.path.append(str(Path.cwd().parent / "src"))
from clear_script import clear_script

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 25)


## 2. Load & clean the data

`clear_script()` encapsulates the full cleaning pipeline: encoding/
separator handling, column name normalization, whitespace stripping,
date parsing, ID typing, categorical conversion, and null-placeholder
unification. See `src/clear_script.py` for the implementation.

In [3]:
input_path = RAW_DIR / "ADQUISICION_DE_BIENES_Y_SERVICIOS_ENERO_A_MARZO_2026.csv"

df = clear_script(str(input_path))

print(f"Cleaned dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


Cleaned dataset shape: 2,498 rows x 19 columns


,ano_eje,sec_ejec,mes_calend,tipo_bien,nro_orden,fecha_orden,nro_ruc,nombre_prov,tipo_proceso,descripcion,descripcion_abreviada,nro_proc_sel,concepto,exp_siaf,fuente_financ,clasificador,nombre_clasificador,total_fact_moneda,fase_orden
0,2026,1028,1,B,1,2026-01-23,20605776478,ALIGMOR S.A.C,106,CONTRATO MENOR,CM,<NA>,ADQUISICION DE SEÑALES Y EQUIPOS DE SEGURIDAD ...,49,18,2.3. 1 11. 1 1,PARA EDIFICIOS Y ESTRUCTURAS,750.0,Compromiso
1,2026,1028,1,B,1,2026-01-23,20605776478,ALIGMOR S.A.C,106,CONTRATO MENOR,CM,<NA>,ADQUISICION DE SEÑALES Y EQUIPOS DE SEGURIDAD ...,49,18,2.6. 3 2. 9 3,SEGURIDAD INDUSTRIAL,5340.0,Compromiso
2,2026,1028,1,B,1,2026-01-23,20605776478,ALIGMOR S.A.C,106,CONTRATO MENOR,CM,<NA>,ADQUISICION DE SEÑALES Y EQUIPOS DE SEGURIDAD ...,49,18,2.3. 1 5. 4 1,"ELECTRICIDAD, ILUMINACION Y ELECTRONICA",2596.0,Compromiso
3,2026,1028,1,B,1,2026-01-23,20605776478,ALIGMOR S.A.C,106,CONTRATO MENOR,CM,<NA>,ADQUISICION DE SEÑALES Y EQUIPOS DE SEGURIDAD ...,49,18,2.3. 1 6. 1 4,DE SEGURIDAD,3600.0,Compromiso
4,2026,1028,1,B,1,2026-01-23,20605776478,ALIGMOR S.A.C,106,CONTRATO MENOR,CM,<NA>,ADQUISICION DE SEÑALES Y EQUIPOS DE SEGURIDAD ...,49,18,2.3. 1 99. 1 99,OTROS BIENES,500.0,Compromiso


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2498 entries, 0 to 2497
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   ano_eje                2498 non-null   int64   
 1   sec_ejec               2498 non-null   int64   
 2   mes_calend             2498 non-null   int64   
 3   tipo_bien              2498 non-null   category
 4   nro_orden              2498 non-null   int64   
 5   fecha_orden            2498 non-null   object  
 6   nro_ruc                2498 non-null   object  
 7   nombre_prov            2498 non-null   object  
 8   tipo_proceso           2498 non-null   category
 9   descripcion            2498 non-null   object  
 10  descripcion_abreviada  2498 non-null   object  
 11  nro_proc_sel           144 non-null    object  
 12  concepto               2498 non-null   object  
 13  exp_siaf               2498 non-null   int64   
 14  fuente_financ          2498 non-null   i

## 3. Data quality validation

This section does not perform any new cleaning — it **verifies**, with
numbers, that the pipeline behaved as expected before the dataset is
considered ready for analysis.

In [5]:
# --- Check 1: leading/trailing whitespace in provider names ---
whitespace_issues = (df["nombre_prov"] != df["nombre_prov"].str.strip()).sum()
assert whitespace_issues == 0, "Unexpected whitespace remaining in nombre_prov"
print(f"[OK] Rows with stray whitespace in nombre_prov: {whitespace_issues}")


[OK] Rows with stray whitespace in nombre_prov: 0


In [ ]:
# --- Check 2: null-placeholder unification in nro_proc_sel ---
null_count = df["nro_proc_sel"].isna().sum()
null_pct = null_count / len(df) * 100
print(f"[INFO] Missing values in nro_proc_sel: {null_count:,} of {len(df):,} rows ({null_pct:.1f}%)")
print("       Expected: this field is only populated for procurement types")
print("       that require a formal selection process (e.g. not for 'Contrato Menor').")


In [ ]:
# --- Check 3: negative amounts must be limited to Rebaja / Anulado phases ---
unexpected_negatives = (
    (df["total_fact_moneda"] < 0)
    & (~df["fase_orden"].isin(["Rebaja", "Anulado"]))
)
assert unexpected_negatives.sum() == 0, "Found unexplained negative amounts"
print(f"[OK] Negative amounts outside Rebaja/Anulado: {unexpected_negatives.sum()}")

negatives_by_phase = df.loc[df["total_fact_moneda"] < 0, "fase_orden"].value_counts()
print("\nNegative amount breakdown by order phase:")
print(negatives_by_phase.to_string())


## 4. Exploratory visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

regular_amounts = df.loc[df["total_fact_moneda"] > 0, "total_fact_moneda"]
ax.boxplot(regular_amounts, vert=True)
ax.set_title("Distribution of order amounts (Compromiso phase only)")
ax.set_ylabel("Amount (PEN)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

total_by_phase = df.groupby("fase_orden", observed=True)["total_fact_moneda"].sum().sort_values()
total_by_phase.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Total amount by order phase")
ax.set_xlabel("Amount (PEN)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

monthly_spend = df.groupby("mes_calend")["total_fact_moneda"].sum()
monthly_spend.plot(kind="bar", ax=ax, color="#55A868")
ax.set_title("Total spend by month — Q1 2026")
ax.set_xlabel("Month")
ax.set_ylabel("Amount (PEN)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


## 5. Export cleaned dataset

In [ ]:
output_path = PROCESSED_DIR / "adquisicion_bienes_2026_q1_clean.csv"
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")


## Summary

| Check | Result |
|---|---|
| Whitespace in `nombre_prov` | Passed — 0 rows affected |
| Null unification in `nro_proc_sel` | Standardized (`.` and `0` treated as `NaN`) |
| Negative amounts outside Rebaja/Anulado | Passed — 0 unexplained rows |

The dataset is considered clean and ready for downstream analysis.